In [ ]:
# --- Welcome to Joe's Kinetic Analyser 3000 --- #

#This tool kit is capable of extracting rate constants from raw data, and the following notebook can model the impacts of descriptor variables on rate constants#
# Follow the steps in the cell below, and read the Notes
# --- Imports --- #
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, KFold, cross_val_predict
from sklearn.metrics import r2_score

In [ ]:
# --- User Changed Settings Go Here --- #

path = r""      #input folder directory with raw kinetics data CSV files 
file_prefix = "prefix_"         #input the CSV file prefix name
time_column = "time"        #input time column name from CSVs
conc_column = "conc"        #input concentration column name from CSV
A0 = 0.005                 #input your initial concentration
N_runs = 10                #input the number of runs
order = 1                   #input the order of reaction here, change until curve is fit well
measured_species = "A"    #input if reactant (A) or product (P) is monitored

# --- Notes --- #
#Time units should be in seconds in the CSV
#concentration should be in M
# [A] pr [P] now included
#[A0] still hardcoded, will add as a parameter to fit

In [ ]:
# --- Read Each CSV File --- #
dfs = {}

for i in range(1,N_runs +1):
    file_name = f"{file_prefix}{i}.csv"
    path_csv = os.path.join(path, file_name)
    dfs[f"df_{i}"] = pd.read_csv(path_csv)

full_df = dfs["df_1"][[time_column, conc_column]].copy()
full_df = full_df.rename(columns = {conc_column: "Run_1"})

for n in range(2, N_runs + 1):
    add_df = dfs[f"df_{n}"][[time_column, conc_column]].copy()
    add_df = add_df.rename(columns={conc_column: f"Run_{n}"})
    full_df = pd.merge(full_df, add_df, on=time_column, how="outer")

print(full_df)
    

In [ ]:
# --- Plot each Kinetic run --- #
for i in range (1, N_runs + 1):
    plt.figure()
    plt.scatter(full_df[time_column], full_df[f"Run_{i}"], color = 'blue')
    plt.title(f"Run_{i}")
    plt.xlabel("time /s")
    plt.ylabel("conc /M")
    plt.show()

In [ ]:
# --- NLs Fit --- # 

def zero_order(time, k):                                  
    return np.minimum(k * time, A0)

def first_order(time, k):
    return A0 * (1 - np.exp(-k * time))

def second_order(time, k):
    return A0 - (A0 / (1 + A0 * k * time))


def order_choice(order):
    if order == 0:
        return zero_order
    elif order == 1:
        return first_order
    elif order == 2:
        return second_order
    else:
        print("order not allowed, please use 0, 1, or 2") 

def init_k(time, conc, order):
    i_50 = (conc-A0/2).abs().idxmin()
    t = time[i_50]
    P = conc[i_50]
    if order == 0:
        return P/t
    elif order == 1:
        return -np.log(1-P/A0)/t
    elif order ==2:
        return P/(A0*(A0-P)*t)


f = order_choice(order)
all_k_values = []

for i in range(1,N_runs+1):
    Working_df = full_df[[time_column,f"Run_{i}"]].dropna()
    x = Working_df[time_column]
    if measured_species == "P":
        y = Working_df[f"Run_{i}"]
    elif measured_species == "A":
        y = A0 - Working_df[f"Run_{i}"]

    k_guess = init_k(x, y, order)
    nls = curve_fit(f, x, y, p0=[k_guess])
    k_est_array = nls[0]
    k_est = k_est_array[0]
    all_k_values.append({'Run': f"Run_{i}",
                         f"k": k_est})

table = pd.DataFrame(all_k_values)
table

# --- This writes your k data into a CSV for future use --- #
#table.to.csv(os.path.join(path "k_values.csv", index = False)
    

In [ ]:
# --- The Plotter 3000 --- #


for i in range(1,N_runs + 1):
    X = full_df[time_column]
    if measured_species == "P":
        y = full_df[f"Run_{i}"]
    elif measured_species == "A":
        y = full_df[f"Run_{i}"]
    k_fit = table.loc[i-1, "k"]
    x_smooth =np.linspace(0, full_df[time_column].max(), num = 1000)
    

    plt.figure()
    plt.scatter(X, y, color='blue')
    if measured_species == "P":
        plt.plot(x_smooth, f(x_smooth, k_fit))
    elif measured_species == "A":
        plt.plot(x_smooth, A0 - f(x_smooth, k_fit))
    plt.title(f"Run_{i}, k = {k_fit:.3g}")
    plt.xlabel("time /s")
    plt.ylabel("conc /M")
    plt.show()